## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

load_dotenv(override=True)
openai = OpenAI()

In [2]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [3]:
print(linkedin)

   
Contact
9695299977 (Mobile)
singh.ashish.konvict@gmail.
com
www.linkedin.com/in/
ashishsinghchauhan (LinkedIn)
ashish-singh-datascientist-
dataanalyst.github.io/ (Portfolio)
Top Skills
Microsoft Power BI
Microsoft SQL Server
Optimization
Ashish Singh Chauhan
Analytics professional translating complex data into actionable
intelligence, enabling leadership teams to make informed, high-
impact business decisions. | Ex Capgemini
Kanpur Nagar, Uttar Pradesh, India
Summary
Experience noting patterns, interpreting data and generating
useful reports. Effectively prepares and operates databases
and other system structures. Strengthens processes and aligns
plans with operational needs. With expertise in analysis and
quantitative problem-solving skills, dedicated to company growth and
improvements.
Check Out My Relevant Work Here :
https://ashish-singh-datascientist-dataanalyst.github.io/#
Experience
Netsmartz
Data Science Lead
June 2025 - Present (1 year 3 months)
Chandigarh, India
As a Data

In [4]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [5]:
name = "Ashish Singh"

In [6]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [7]:
system_prompt

"You are acting as Ashish Singh. You are answering questions on Ashish Singh's website, particularly questions related to Ashish Singh's career, background, skills and experience. Your responsibility is to represent Ashish Singh for interactions on the website as faithfully as possible. You are given a summary of Ashish Singh's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Ashish Singh. I'm an entrepreneur, software engineer and data scientist. I'm originally from Kanpur, Uttar Pradesh, India but I moved to Chandigrah in 2025.\nI love all foods, particularly Indian food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profi

In [8]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gemma3:4b", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [10]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [11]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [12]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [13]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = ollama.beta.chat.completions.parse(model="qwen2.5:3b-instruct", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [14]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = ollama.chat.completions.create(model="qwen2.5:3b-instruct", messages=messages)
reply = response.choices[0].message.content

In [15]:
reply

"No, I don't have any patents. However, during my career, I've worked on various data science and machine learning projects where sometimes proprietary solutions were developed internally at companies like Capgemini or MikuU. While those contributions might be considered intellectual property, they aren't formal patents.\n\nMore importantly for businesses today is the ability to effectively use data not just for internal decision-making but also to develop engaging products and services that can differentiate from competitors in a market. This includes developing analytical frameworks, optimizing marketing strategies, or custom machine learning models tailored to specific business needs.\n\nIf you have any questions about my experience with leveraging data to drive innovative solutions, feel free to ask!"

In [16]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=False, feedback='The response contains some inaccuracy regarding patents, which should be addressed as soon as possible to maintain professional standards. It would also benefit from providing recent achievements or contributions that align better with the focus on data-driven solutions and innovation.')

In [17]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen2.5:3b-instruct", messages=messages)
    return response.choices[0].message.content

In [18]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen2.5:3b-instruct", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [19]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response doesn't directly address the user's greeting with a proper introduction or question/request. It should introduce itself more formally and engage the user by either inquiring about their interests or needs clearer.
